# 3 · Profiles and statistics

Radial and angular profiles, Monte-Carlo maps, cold-electron production and temperature –
the analysis figures from the second half of `asymmetricPlots.m`.

Every plotting function returns `(fig, axes, data)`; `data` contains the plotted numbers
(the MATLAB `prof` / `out` / `summary` structs) so they can be reused or exported.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

import collide_comet as cc
from collide_comet import plotting as ccp
from run_config import get_cases, make_saver

cases, DEMO = get_cases()
save = make_saver("profiles", DEMO)
cases

## Radial profiles

`make_electrondens_radial_AllCases_Plot`: volume-weighted means in spherical shells.

In [ ]:
fig, axes, profiles = ccp.radial_profiles_plot(cases, r_edges=np.arange(0, 305, 5), r_max=300)
save(fig, "densities_vs_r");

In [ ]:
# the numbers behind the figure
ref = profiles["Ref"]
ok = np.isfinite(ref["mean_total"])        # shells without any cell centre are NaN
for r, n_tot, n_cold in list(zip(ref["r_centres"][ok], ref["mean_total"][ok], ref["mean_cold"][ok]))[:6]:
    print(f"r = {r:6.1f} km   n_e = {n_tot:8.3f} cm^-3   n_cold = {n_cold:8.3f} cm^-3")

## Angular profiles in a shell

`make_electrondens_angleToQVec_shell_MC_AllCases_Plot_v2`: Monte-Carlo points in the shell
$r_0 \pm \Delta r/2$, binned by angle to the direction of maximum outgassing. The reference case is
binned about each asymmetric case's axis (dashed). The random points use the same seeded generator
as MATLAB's `rng(1)`.

In [ ]:
fig, axes, shell = ccp.angle_shell_profiles_plot(cases, r0=50, dr=20, n_mc=500_000, n_angle_bins=20)
save(fig, "densities_vs_angle_r50");

## Radius–angle maps

`make_electrondens_rAngle_QVec_2d_MC_AllCases_Plot_v2` for the total density, cold density and cold fraction. The reference case is drawn first (top left), relative to +X, +Y and +Z; the dashed line is the estimated electron exobase distance (`show_exobase=False` turns it off). `method="cells"` bins the grid cells directly instead of Monte-Carlo sampling.

In [ ]:
for quantity in ["total", "cold", "fraction"]:
    fig, axes, maps = ccp.r_angle_maps_plot(
        cases, quantity=quantity, statistic="median", n_mc=300_000,
        n_rad_bins=30, n_angle_bins=20, r_min=0, r_max=300,
    )
    save(fig, f"r_angle_map_{quantity}")

## Profiles along the six semi-axes

`make_electrondens_axisCone_MC_scaledExobase_Plot` for one case: 10° cones about ±X, ±Y, ±Z,
optionally with distance scaled by the local exobase distance.

In [ ]:
case = cases["+Z"]
fig, axes, cones = ccp.axis_cone_profiles_plot(case, n_mc=200_000, scale_by_local_exobase=False)
save(fig, f"axis_cones_case{case.name}")
fig, axes, cones = ccp.axis_cone_profiles_plot(case, n_mc=200_000, scale_by_local_exobase=True)
save(fig, f"axis_cones_scaled_case{case.name}");

## Cold-electron production

`make_grouped_cold_e_prod_Plot_v3` (maps + shell summary) and the volume-weighted histogram.

In [ ]:
fig, axes, summary = ccp.cold_production_summary_plot(cases, view="XZ", r0=50, dr=10, n_angle_bins=36)
save(fig, "cold_production_XZ_summary");

In [ ]:
fig, ax, hist = ccp.cold_production_histogram(
    cases, n_bins=50, normalization="probability", r_range=(0, 55),
    yscale="log", y_min_log=1e-5,          # log y axis; empty bins are not drawn
    percentiles=(50, 75, 90),              # volume-weighted median, 75th and 90th percentile lines
    print_percentiles=True,
)
save(fig, "cold_production_histogram_r0-55")

## Temperature versus cometocentric distance

`makeTemperature_vs_r_Plot_v2` for the reference case, with the `rat22` fit.

In [ ]:
fig, ax, tvr = ccp.temperature_vs_radius_plot(cases.reference, mode="combined")
save(fig, "temperature_vs_r_ref")
if tvr["fit_params"] is not None:
    p1, p2, p3, q1, q2 = tvr["fit_params"]
    print(f"T(r) = ({p1:.4g} r^2 + {p2:.4g} r + {p3:.4g}) / (r^2 + {q1:.4g} r + {q2:.4g})")